In [9]:
!apt-get update -qq && apt-get install -y ffmpeg mediainfo cmake build-essential
!pip install -q openai-whisper sentence-transformers gradio Pillow pymediainfo ultralytics face_recognition deep-translator
!mkdir -p /content/known_faces

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
mediainfo is already the newest version (24.01.1-1build2).
cmake is already the newest version (3.28.3-1build7).
build-essential is already the newest version (12.10ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.


In [ ]:
import os
import re
import json
import sqlite3
import hashlib
import subprocess
import time
import torch
import numpy as np
import cv2
from PIL import Image
from google.colab import drive
import whisper
from sentence_transformers import SentenceTransformer
from ultralytics import YOLO
import face_recognition
from deep_translator import GoogleTranslator
import gradio as gr

# 1. Mount Google Drive & Configure Storage
drive.mount('/content/drive', force_remount=False)

BASE_VAULT = "/content/drive/MyDrive/MAM_Production_Vault"
PROXIES_DIR = os.path.join(BASE_VAULT, "proxies")
THUMBS_DIR = os.path.join(BASE_VAULT, "thumbnails")
KNOWN_FACES_DIR = "/content/known_faces"
DB_PATH = os.path.join(BASE_VAULT, "multimodal_mam.db")

for p in [PROXIES_DIR, THUMBS_DIR, KNOWN_FACES_DIR]:
    os.makedirs(p, exist_ok=True)

# 2. Hardware Acceleration & AI Models Initialization
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing AI models on {DEVICE}...")

# Speech-to-English translation engine
whisper_model = whisper.load_model("small", device=DEVICE)

# Dedicated semantic similarity model for English QA matching
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

# Fast lightweight object detection engine
yolo_model = YOLO("yolov8n.pt")

# Hinglish / Multilingual query normalizer
query_translator = GoogleTranslator(source='auto', target='en')

# 3. Known Face Registry Initialization
known_face_encodings = []
known_face_names = []

def reload_known_faces():
    global known_face_encodings, known_face_names
    known_face_encodings = []
    known_face_names = []
    if os.path.exists(KNOWN_FACES_DIR):
        for f in os.listdir(KNOWN_FACES_DIR):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                name = os.path.splitext(f)[0]
                img_p = os.path.join(KNOWN_FACES_DIR, f)
                try:
                    img = face_recognition.load_image_file(img_p)
                    encs = face_recognition.face_encodings(img)
                    if encs:
                        known_face_encodings.append(encs[0])
                        known_face_names.append(name)
                except Exception as e:
                    print(f"[!] Warning reading face '{f}': {e}")
    print(f"[*] Loaded {len(known_face_names)} reference faces: {known_face_names}")

reload_known_faces()

# 4. Database Setup with Write-Ahead Logging (WAL)
def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode=WAL;")
    return conn

def init_db():
    with get_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS assets (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                file_hash TEXT UNIQUE,
                filename TEXT,
                drive_path TEXT,
                proxy_path TEXT,
                thumb_path TEXT,
                duration REAL,
                resolution TEXT,
                codec TEXT,
                audio_codec TEXT,
                bitrate_kbps INTEGER,
                fps REAL,
                file_size_mb REAL,
                detected_lang TEXT,
                full_english_transcript TEXT,
                synopsis TEXT,
                keywords TEXT,
                visual_objects TEXT,
                visual_faces TEXT,
                audio_entities TEXT,
                ingest_sec REAL
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS transcript_segments (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                asset_id INTEGER,
                start_sec REAL,
                end_sec REAL,
                english_text TEXT,
                embedding_json TEXT,
                FOREIGN KEY(asset_id) REFERENCES assets(id)
            )
        """)

init_db()

# 5. Technical Metadata Extraction via FFprobe
def extract_tech_metadata(file_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "stream=codec_name,codec_type,width,height,r_frame_rate,duration,bit_rate:format=duration,size,bit_rate",
        "-of", "json", file_path
    ]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    v_codec, a_codec = "Unknown", "Unknown"
    width, height = 0, 0
    duration, fps, bitrate = 0.0, 0.0, 0
    try:
        data = json.loads(res.stdout)
        for s in data.get("streams", []):
            if s.get("codec_type") == "video" and width == 0:
                width = s.get("width", 0)
                height = s.get("height", 0)
                v_codec = s.get("codec_name", "unknown")
                if "r_frame_rate" in s and "/" in s["r_frame_rate"]:
                    n, d = map(float, s["r_frame_rate"].split("/"))
                    fps = round(n / d, 2) if d > 0 else 0.0
            elif s.get("codec_type") == "audio" and a_codec == "Unknown":
                a_codec = s.get("codec_name", "unknown")
        fmt = data.get("format", {})
        duration = float(fmt.get("duration", 0.0))
        bitrate = int(int(fmt.get("bit_rate", 0)) / 1000)
    except Exception:
        pass
    return v_codec, a_codec, f"{width}x{height}" if width and height else "Unknown", round(duration, 2), fps, bitrate

# 6. Video Track Visual Analysis (YOLO Objects + Deep Metric Faces)
def analyze_video_visuals(video_path, sample_interval_sec=2.0):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frame_step = max(1, int(fps * sample_interval_sec))

    detected_objects = set()
    detected_faces = set()
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_step == 0:
            yolo_results = yolo_model(frame, verbose=False, device=0 if DEVICE == "cuda" else "cpu")[0]
            for box in yolo_results.boxes:
                cls_name = yolo_model.names[int(box.cls[0])]
                if cls_name != "person":
                    detected_objects.add(cls_name)

            small_rgb = cv2.cvtColor(cv2.resize(frame, (0, 0), fx=0.5, fy=0.5), cv2.COLOR_BGR2RGB)
            face_locations = face_recognition.face_locations(small_rgb)
            if face_locations:
                encs = face_recognition.face_encodings(small_rgb, face_locations)
                for enc in encs:
                    match_name = "Unknown Face"
                    if known_face_encodings:
                        matches = face_recognition.compare_faces(known_face_encodings, enc, tolerance=0.55)
                        if True in matches:
                            match_name = known_face_names[matches.index(True)]
                    detected_faces.add(match_name)

        frame_idx += 1

    cap.release()
    return list(detected_objects), list(detected_faces)

# 7. Audio Entity Extraction (Public Figures & Topics Mentioned in Speech)
def extract_audio_entities(transcript_text):
    known_entities = [
        "Narendra Modi", "Modi", "Rahul Gandhi", "Amit Shah", "Arvind Kejriwal",
        "Parliament", "Supreme Court", "Budget", "Lok Sabha", "Rajya Sabha",
        "Police", "Military", "Doctor", "Journalist", "Election", "Car", "Airplane"
    ]
    found = []
    for ent in known_entities:
        if re.search(r'\b' + re.escape(ent) + r'\b', transcript_text, re.IGNORECASE):
            found.append(ent)
    return list(dict.fromkeys(found))

# 8. Single Video Ingestion Pipeline
def process_single_video(video_path):
    filename = os.path.basename(video_path)
    file_size_mb = round(os.path.getsize(video_path) / (1024 * 1024), 2)

    hasher = hashlib.sha256()
    with open(video_path, 'rb') as f:
        while chunk := f.read(4 * 1024 * 1024):
            hasher.update(chunk)
    file_hash = hasher.hexdigest()
    uid = file_hash[:8]

    with get_db() as conn:
        row = conn.execute("SELECT id FROM assets WHERE file_hash = ?", (file_hash,)).fetchone()
        if row:
            return f"Skipped (Already Ingested): {filename}"

    t0 = time.time()
    v_codec, a_codec, resolution, duration, fps, bitrate = extract_tech_metadata(video_path)

    proxy_path = os.path.join(PROXIES_DIR, f"{uid}_proxy.mp4")
    thumb_path = os.path.join(THUMBS_DIR, f"{uid}_thumb.jpg")

    subprocess.run([
        "ffmpeg", "-y", "-nostdin", "-ss", "00:00:02", "-i", video_path,
        "-vframes", "1", "-q:v", "2", thumb_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if "avc" in v_codec or "h264" in v_codec:
        subprocess.run([
            "ffmpeg", "-y", "-nostdin", "-i", video_path,
            "-c", "copy", "-movflags", "+faststart", proxy_path
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    else:
        subprocess.run([
            "ffmpeg", "-y", "-nostdin", "-i", video_path,
            "-vf", "scale=-2:720", "-c:v", "libx264", "-preset", "ultrafast",
            "-c:a", "aac", "-b:a", "96k", "-movflags", "+faststart", proxy_path
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    vis_objects, vis_faces = analyze_video_visuals(video_path, sample_interval_sec=2.0)

    temp_wav = os.path.join(THUMBS_DIR, f"{uid}_audio.wav")
    subprocess.run([
        "ffmpeg", "-y", "-nostdin", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", temp_wav
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    segments_data = []
    full_english_text = ""
    detected_lang = "Unknown"

    if os.path.exists(temp_wav) and os.path.getsize(temp_wav) > 1000:
        try:
            whisper_out = whisper_model.transcribe(
                temp_wav,
                task="translate",
                fp16=(DEVICE == "cuda")
            )
            detected_lang = whisper_out.get("language", "Unknown")
            segments = whisper_out.get("segments", [])

            clean_texts = [s["text"].strip() for s in segments if len(s["text"].strip()) > 3]
            if clean_texts:
                embeddings = embed_model.encode(clean_texts, convert_to_numpy=True, batch_size=32)
                for s, emb in zip(segments, embeddings):
                    segments_data.append({
                        "start": round(s["start"], 2),
                        "end": round(s["end"], 2),
                        "text": s["text"].strip(),
                        "embedding": emb.tolist()
                    })
                full_english_text = " ".join(clean_texts)
        except Exception as e:
            full_english_text = f"Audio processing notice: {str(e)}"
        finally:
            if os.path.exists(temp_wav):
                os.remove(temp_wav)

    audio_entities = extract_audio_entities(full_english_text)
    stopwords = {"this", "that", "with", "from", "have", "were", "they", "your", "what", "about", "there", "their", "will"}
    tokens = [w.strip(".,!?:;\"'()[]").lower() for w in full_english_text.split() if len(w) > 3]
    top_keywords = list(dict.fromkeys([w for w in tokens if w not in stopwords]))[:15]
    keywords_str = ", ".join(top_keywords)

    synopsis = " ".join(full_english_text.split(". ")[:3]).strip()
    if synopsis and not synopsis.endswith("."):
        synopsis += "."
    if not synopsis:
        synopsis = "No dialogue detected to generate synopsis."

    elapsed = round(time.time() - t0, 2)

    with get_db() as conn:
        cursor = conn.execute("""
            INSERT INTO assets (
                file_hash, filename, drive_path, proxy_path, thumb_path,
                duration, resolution, codec, audio_codec, bitrate_kbps, fps,
                file_size_mb, detected_lang, full_english_transcript, synopsis,
                keywords, visual_objects, visual_faces, audio_entities, ingest_sec
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            file_hash, filename, video_path, proxy_path, thumb_path,
            duration, resolution, v_codec, a_codec, bitrate, fps,
            file_size_mb, detected_lang.upper(), full_english_text, synopsis,
            keywords_str, ", ".join(vis_objects), ", ".join(vis_faces), ", ".join(audio_entities), elapsed
        ))
        asset_id = cursor.lastrowid

        for seg in segments_data:
            conn.execute("""
                INSERT INTO transcript_segments (asset_id, start_sec, end_sec, english_text, embedding_json)
                VALUES (?, ?, ?, ?, ?)
            """, (asset_id, seg["start"], seg["end"], seg["text"], json.dumps(seg["embedding"])))

    return f"Ingested {filename} in {elapsed}s | Lang: {detected_lang.upper()} | Faces: {vis_faces} | Objects: {vis_objects}"

# 9. Batch Google Drive Folder Ingest
def ingest_drive_folder(folder_path):
    if not os.path.exists(folder_path):
        return f"Error: Google Drive path '{folder_path}' does not exist."
    valid_exts = {".mp4", ".mov", ".mkv", ".avi", ".webm"}
    video_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if os.path.splitext(f)[1].lower() in valid_exts]
    if not video_files:
        return f"No video files found in '{folder_path}'."

    reload_known_faces()
    logs = []
    print(f"[*] Found {len(video_files)} files in Drive folder. Starting batch processing...")
    for idx, v_path in enumerate(video_files, 1):
        msg = process_single_video(v_path)
        logs.append(f"[{idx}/{len(video_files)}] {msg}")
    return "\n".join(logs)

# 10. High-Precision Hybrid Semantic Search
def normalize_query_to_english(raw_query):
    raw_query = raw_query.strip()
    try:
        translated = query_translator.translate(raw_query)
        return translated
    except Exception:
        return raw_query

def extract_query_keywords(query_str):
    stopwords = {"what", "did", "say", "about", "tell", "show", "where", "when", "how", "the", "in", "is"}
    words = re.findall(r'\b[a-zA-Z]{3,}\b', query_str.lower())
    return [w for w in words if w not in stopwords]

def hybrid_semantic_search(user_query, top_k=6):
    if not user_query.strip():
        return [], "Please enter a search query.", None, None

    english_query = normalize_query_to_english(user_query)
    query_keywords = list(set(extract_query_keywords(english_query) + extract_query_keywords(user_query)))

    query_vector = embed_model.encode(english_query, convert_to_numpy=True)
    norm_q = np.linalg.norm(query_vector)

    with get_db() as conn:
        segments = conn.execute("""
            SELECT s.asset_id, a.filename, a.detected_lang, a.proxy_path, a.thumb_path,
                   s.start_sec, s.end_sec, s.english_text, s.embedding_json,
                   a.synopsis, a.keywords, a.visual_objects, a.visual_faces, a.audio_entities, a.resolution, a.duration
            FROM transcript_segments s
            JOIN assets a ON s.asset_id = a.id
        """).fetchall()

    if not segments:
        return [], "No indexed segments found. Ingest your Drive videos first.", None, None

    scored_results = []
    for row in segments:
        asset_id, fname, lang, proxy_p, thumb_p, start, end, text, emb_json, syn, kws, v_objs, v_faces, a_ents, res, dur = row
        seg_vector = np.array(json.loads(emb_json))
        norm_s = np.linalg.norm(seg_vector)

        dense_score = float(np.dot(query_vector, seg_vector) / (norm_q * norm_s)) if (norm_q and norm_s) else 0.0

        text_lower = text.lower()
        faces_lower = (v_faces or "").lower()
        objs_lower = (v_objs or "").lower()
        ents_lower = (a_ents or "").lower()

        matched_tokens = [k for k in query_keywords if (k in text_lower or k in faces_lower or k in objs_lower or k in ents_lower)]
        lexical_boost = min(len(matched_tokens) * 0.15, 0.40)

        final_score = (dense_score * 0.65) + lexical_boost

        scored_results.append({
            "asset_id": asset_id,
            "filename": fname,
            "lang": lang,
            "proxy_path": proxy_p,
            "thumb_path": thumb_p,
            "timestamp": f"{int(start//60):02d}:{int(start%60):02d} - {int(end//60):02d}:{int(end%60):02d}",
            "text": text,
            "final_score": round(final_score, 3),
            "dense_score": round(dense_score, 3),
            "matched_terms": ", ".join(matched_tokens) if matched_tokens else "Contextual match",
            "synopsis": syn,
            "keywords": kws,
            "visual_objects": v_objs,
            "visual_faces": v_faces,
            "audio_entities": a_ents,
            "resolution": res,
            "duration": dur
        })

    scored_results.sort(key=lambda x: x["final_score"], reverse=True)
    top_matches = scored_results[:top_k]

    table_data = []
    for m in top_matches:
        table_data.append([
            m["asset_id"],
            m["filename"],
            m["timestamp"],
            m["final_score"],
            m["matched_terms"],
            m["visual_faces"],
            m["text"]
        ])

    best = top_matches[0]
    detail_md = f"""### 🎯 Best Search Match
- **User Query:** `{user_query}`
- **Interpreted English Query:** `{english_query}`
- **File:** `{best['filename']}` ({best['resolution']}, {best['duration']}s)
- **Spoken Language:** `{best['lang']}` (Auto-translated to English)
- **Exact Timestamp:** `{best['timestamp']}`
- **Relevance Confidence:** `{best['final_score']}` *(Dense: {best['dense_score']})*

---
### 🎙️ Spoken Statement
> *"{best['text']}"*

---
### 🔍 Multimodal Visual & Audio Detections
- **Visual Faces Identified:** `{best['visual_faces'] or 'None identified'}`
- **Visual Objects Identified:** `{best['visual_objects'] or 'None identified'}`
- **Spoken Entities Identified:** `{best['audio_entities'] or 'None detected'}`

---
### 📝 Video Synopsis & Keywords
**Synopsis:** {best['synopsis']}

**Keywords:** `{best['keywords']}`
"""
    return table_data, detail_md, best["proxy_path"], best["thumb_path"]

# 11. Editable Transcript & Technical Registry Handlers
def load_transcript(asset_id):
    if not asset_id:
        return "", "Provide a valid Asset ID."
    with get_db() as conn:
        row = conn.execute("SELECT full_english_transcript, filename FROM assets WHERE id = ?", (asset_id,)).fetchone()
    return (row[0], f"Loaded transcript for {row[1]}") if row else ("", f"Asset #{asset_id} not found.")

def save_transcript(asset_id, text):
    if not asset_id or not text.strip():
        return "Asset ID and text cannot be empty."
    with get_db() as conn:
        conn.execute("UPDATE assets SET full_english_transcript = ? WHERE id = ?", (text, asset_id))
    return f"Successfully saved updated transcript for Asset #{asset_id}."

def load_tech_table():
    with get_db() as conn:
        return conn.execute("""
            SELECT id, filename, detected_lang, resolution, fps, codec, audio_codec, bitrate_kbps, duration, visual_faces, visual_objects
            FROM assets ORDER BY id DESC
        """).fetchall()

# 12. Interactive Gradio Interface
with gr.Blocks(title="Enterprise Multilingual MAM") as app:
    gr.Markdown("# 🎬 Enterprise Multimodal MAM Console")
    gr.Markdown("Batch Google Drive Ingestion, Hinglish Semantic Search with Exact Timestamps, Audio-Visual Face/Object Identification, and Technical Metadata.")

    with gr.Tab("1. Batch Drive Ingestion"):
        with gr.Row():
            drive_folder = gr.Textbox(
                value="/content/drive/MyDrive/MyVideos",
                label="Google Drive Folder Path",
                placeholder="/content/drive/MyDrive/YourVideosFolder"
            )
            start_ingest_btn = gr.Button("Ingest All Videos", variant="primary")
        ingest_log = gr.Textbox(label="Batch Extraction Progress (Faces, Objects, Translation)", lines=8)
        start_ingest_btn.click(fn=ingest_drive_folder, inputs=[drive_folder], outputs=[ingest_log])

    with gr.Tab("2. Multimodal Semantic Search"):
        with gr.Row():
            search_query = gr.Textbox(
                label="Hinglish / English / Multimodal Query",
                placeholder="e.g. modi ne rahul gandhi ke baare mein kya bola / budget speech / car or vehicle on road"
            )
            run_search_btn = gr.Button("Search Video Library", variant="primary")

        with gr.Row():
            with gr.Column(scale=2):
                matches_table = gr.Dataframe(
                    headers=["Asset ID", "Filename", "Timestamp", "Score", "Matched Tokens", "Visual Faces", "Translated English Dialogue"],
                    datatype=["number", "str", "str", "number", "str", "str", "str"],
                    interactive=False
                )
            with gr.Column(scale=1):
                analysis_card = gr.Markdown()
                preview_player = gr.Video(label="Matching Video Proxy Clip")
                preview_thumb = gr.Image(label="Keyframe Thumbnail")

        run_search_btn.click(
            fn=hybrid_semantic_search,
            inputs=[search_query],
            outputs=[matches_table, analysis_card, preview_player, preview_thumb]
        )

    with gr.Tab("3. Editable English Transcripts"):
        with gr.Row():
            id_input = gr.Number(label="Asset ID", precision=0)
            load_btn = gr.Button("Fetch Transcript")
        status_lbl = gr.Textbox(label="Status", interactive=False)
        editor_area = gr.TextArea(label="Translated English Transcript (Editable)", lines=12)
        save_btn = gr.Button("Save Corrections to Catalog", variant="primary")

        load_btn.click(fn=load_transcript, inputs=[id_input], outputs=[editor_area, status_lbl])
        save_btn.click(fn=save_transcript, inputs=[id_input, editor_area], outputs=[status_lbl])

    with gr.Tab("4. Technical & Visual Catalog"):
        refresh_catalog_btn = gr.Button("Refresh Catalog")
        catalog_grid = gr.Dataframe(
            headers=["ID", "Filename", "Lang", "Resolution", "FPS", "Video Codec", "Audio Codec", "Bitrate (kbps)", "Duration (s)", "Faces", "Objects"],
            datatype=["number", "str", "str", "str", "number", "str", "str", "number", "number", "str", "str"],
            interactive=False
        )
        refresh_catalog_btn.click(fn=load_tech_table, outputs=[catalog_grid])

app.launch(debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[*] Initializing AI models on cuda...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[*] Loaded 0 reference faces: []
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39ff0b8ede8e385bc7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[*] Loaded 0 reference faces: []
[*] Found 10 files in Drive folder. Starting batch processing...
